# Notebook 05: Two Qubits and Tensor Products

So far we have worked with single-qubit states -- vectors in a 2-dimensional
Hilbert space. Real quantum computers operate on *many* qubits at once.  How do
we combine individual qubits into a larger system?

The answer is the **tensor product** (also called the Kronecker product).  When
we combine two 2-dimensional spaces, the resulting space has dimension
$2 \times 2 = 4$.  In general, $n$ qubits live in a $2^n$-dimensional space.

This notebook covers:
1. How the tensor product combines single-qubit states
2. The 4-dimensional computational basis for 2 qubits
3. Labeling basis states with bitstrings
4. Building 2-qubit states and examining their amplitudes and probabilities

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from quantum_demo.linalg import ket, is_normalized
from quantum_demo.states import basis_state, qubit_state, amplitudes_to_probabilities, pretty_basis_labels
from quantum_demo.tensor import tensor, basis_state_bits
from quantum_demo.viz.static_plots import plot_probabilities, plot_real_amplitudes

## 1. Single-qubit building blocks

Recall that a single qubit lives in $\mathbb{C}^2$. The computational basis
states are:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

Let's create these and verify them.

In [ ]:
ket0 = ket(0, 2)
ket1 = ket(1, 2)

print("|0> =", ket0)
print("|1> =", ket1)
print()
print("Both normalized?", is_normalized(ket0), is_normalized(ket1))

## 2. The tensor product: combining two qubits

When qubit A is in state $|\psi_A\rangle$ and qubit B is in state $|\psi_B\rangle$,
the combined system is described by the **tensor product**:

$$|\psi_{AB}\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$$

For column vectors, this is the Kronecker product: every entry of the first
vector multiplied by the entire second vector.

With 2 qubits, each of dimension 2, we get a state vector of dimension
$2 \times 2 = 4$.

Let's compute $|0\rangle \otimes |0\rangle$ explicitly.

In [ ]:
# Tensor product of |0> with |0>
state_00 = tensor(ket0, ket0)
print("|0> (x) |0> =", state_00)
print("Dimension:", len(state_00))
print("Normalized?", is_normalized(state_00))

The result is a 4-dimensional vector $[1, 0, 0, 0]^T$ -- the first
computational basis vector in the 4D space.

Let's see all four 2-qubit computational basis states.

In [ ]:
# All 2-qubit computational basis states via tensor product
state_01 = tensor(ket0, ket1)
state_10 = tensor(ket1, ket0)
state_11 = tensor(ket1, ket1)

print("|00> =", state_00)
print("|01> =", state_01)
print("|10> =", state_10)
print("|11> =", state_11)

Each basis state is a one-hot vector in $\mathbb{C}^4$.  The ordering follows
the standard convention where the bitstring is read as a binary number:

| State | Binary | Index |
|-------|--------|-------|
| $|00\rangle$ | 00 | 0 |
| $|01\rangle$ | 01 | 1 |
| $|10\rangle$ | 10 | 2 |
| $|11\rangle$ | 11 | 3 |

## 3. Shortcut: `basis_state_bits`

Instead of manually tensoring individual qubit states, `basis_state_bits`
accepts a bitstring and builds the tensor product for you.

In [ ]:
# Shorthand: directly from bitstrings
for bits in ['00', '01', '10', '11']:
    state = basis_state_bits(bits)
    print(f"|{bits}> = {state}")

In [ ]:
# Verify it matches the manual tensor product
assert np.allclose(basis_state_bits('10'), tensor(ket1, ket0))
print("basis_state_bits('10') matches tensor(|1>, |0>): confirmed!")

## 4. Pretty basis labels

When we plot states, we need labels like `'00'`, `'01'`, `'10'`, `'11'` for the
bars.  The function `pretty_basis_labels` generates these from the number of
qubits.

In [ ]:
labels_2q = pretty_basis_labels(2)
print("2-qubit labels:", labels_2q)

labels_3q = pretty_basis_labels(3)
print("3-qubit labels:", labels_3q)

## 5. Tensor products of superpositions

Things get more interesting when we tensor together *superposition* states.

Consider two qubits, each in the $|+\rangle$ state:

$$|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$$

Their tensor product is:

$$|+\rangle \otimes |+\rangle = \frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$$

This is a **uniform superposition** over all 4 basis states.

In [ ]:
# |+> state
plus = qubit_state(1, 1)  # normalize(|0> + |1>)
print("|+> =", plus)

# Tensor product |+> (x) |+>
plus_plus = tensor(plus, plus)
print("|+> (x) |+> =", plus_plus)
print("Normalized?", is_normalized(plus_plus))

In [ ]:
# Visualize the amplitudes
labels = pretty_basis_labels(2)
fig = plot_real_amplitudes(plus_plus, labels=labels, title="|+> (x) |+>: Amplitudes")
fig

In [ ]:
# And the probabilities
probs = amplitudes_to_probabilities(plus_plus)
print("Probabilities:", probs)
print("Sum:", probs.sum())

fig = plot_probabilities(probs, labels=labels, title="|+> (x) |+>: Probabilities")
fig

Each outcome has equal probability $1/4 = 0.25$. If we measured both qubits,
we would see `00`, `01`, `10`, or `11` each about 25% of the time.

## 6. Non-uniform 2-qubit states

Now let's try tensoring two *different* qubit states to build a non-uniform
2-qubit state.

Suppose qubit A is $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and qubit B
is biased toward $|0\rangle$, say $\sqrt{0.9}\,|0\rangle + \sqrt{0.1}\,|1\rangle$.

In [ ]:
# Qubit A: equal superposition
qubitA = qubit_state(1, 1)

# Qubit B: biased toward |0>
qubitB = qubit_state(np.sqrt(0.9), np.sqrt(0.1))

# Combined 2-qubit state
state_AB = tensor(qubitA, qubitB)

print("Qubit A:", qubitA)
print("Qubit B:", qubitB)
print("Combined:", state_AB)
print("Normalized?", is_normalized(state_AB))

In [ ]:
# Examine amplitudes and probabilities side by side
probs_AB = amplitudes_to_probabilities(state_AB)
labels = pretty_basis_labels(2)

print("Amplitudes:")
for lbl, amp in zip(labels, state_AB):
    print(f"  |{lbl}>: {amp.real:+.4f}")

print("\nProbabilities:")
for lbl, p in zip(labels, probs_AB):
    print(f"  |{lbl}>: {p:.4f}")
print(f"  Sum: {probs_AB.sum():.4f}")

In [ ]:
fig = plot_probabilities(probs_AB, labels=labels,
                         title="Non-uniform 2-qubit state: Probabilities")
fig

Notice how the 2-qubit probabilities factor into products of the individual
qubit probabilities.  For example, $P(01) = P_A(0) \times P_B(1) = 0.5 \times 0.1 = 0.05$.

This is a hallmark of **product states** -- states that can be written as
a tensor product of individual qubit states.  Not all 2-qubit states have this
property; those that don't are called **entangled** states. We will see
entanglement when we introduce the CNOT gate in the next notebook.

## 7. Scaling up: 3 qubits

The tensor product extends to any number of qubits. Three qubits live in
$2^3 = 8$ dimensions.

In [ ]:
# Three-qubit example: |+> (x) |0> (x) |1>
state_3q = tensor(plus, ket0, ket1)
labels_3 = pretty_basis_labels(3)

print("3-qubit state dimension:", len(state_3q))
print()
for lbl, amp, prob in zip(labels_3, state_3q, amplitudes_to_probabilities(state_3q)):
    if abs(amp) > 1e-10:
        print(f"  |{lbl}>: amplitude = {amp.real:+.4f}, probability = {prob:.4f}")

In [ ]:
# Visualize
probs_3q = amplitudes_to_probabilities(state_3q)
fig = plot_probabilities(probs_3q, labels=labels_3,
                         title="3-qubit state |+>(x)|0>(x)|1>")
fig

Only two basis states have non-zero amplitude -- `001` and `101` -- because
the second qubit is fixed at $|0\rangle$ and the third at $|1\rangle$.  Only the
first qubit is in superposition.

This confirms the exponential growth pattern:

| Qubits | Dimension |
|--------|-----------|
| 1      | 2         |
| 2      | 4         |
| 3      | 8         |
| 10     | 1,024     |
| 20     | 1,048,576 |
| 30     | ~10^9     |

This exponential scaling is both the promise and the challenge of quantum
computing: the state space grows incredibly fast, which is why simulating
quantum systems on classical computers becomes intractable.

## Key takeaways

- The **tensor product** $\otimes$ combines individual qubit Hilbert spaces into a joint space.
- Two qubits produce a 4-dimensional space; $n$ qubits produce a $2^n$-dimensional space.
- Computational basis states are labeled by bitstrings: $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.
- `basis_state_bits('10')` constructs the tensor product $|1\rangle \otimes |0\rangle$ from a bitstring.
- `pretty_basis_labels(n)` generates the corresponding labels for plotting.
- Product states have probabilities that factor; entangled states (coming next!) do not.

**Next notebook:** We will apply quantum gates (H, X, Z, CNOT) to these
multi-qubit states and see how unitarity preserves norms and how phase flips
enable interference.